# SpatialWarp tutorial

This notebook walks through everything `spatialwarp` does, using synthetic data so it runs
end-to-end with no external files. For a real dataset (MSI + Visium HD), see
`examples/l12_walkthrough.ipynb`.

`spatialwarp` solves two separate problems, both pure Python (no Fiji/BUnwarpJ/JVM):

1. **Grid self-alignment** (`spatialwarp.grid_align`) — some spatial technologies (e.g. MSI
   instrument rasters) report spot coordinates in a system that doesn't share an
   origin/scale/rotation with their own reference image. This rasterizes the point/intensity
   data into a pseudo-image and registers it onto the real image.
2. **Cross-modality registration** (`spatialwarp.registration`, `spatialwarp.pipeline`) —
   aligning two *different* images (e.g. an MSI section's H&E and a Visium slide's H&E) via
   landmark-guided elastic (B-spline) registration through SimpleITK, then warping one
   dataset's points into the other's space and nearest-neighbor matching them.

Both problems turn out to use the same underlying machinery — grid self-alignment is just
cross-modality registration where one side is a rasterized point cloud instead of a real image.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import spatialwarp
from spatialwarp.grid_align import rasterize_points, points_to_pixel, run_grid_alignment
from spatialwarp.landmark_picker import pick_landmarks
from spatialwarp.registration import register_elastic, RegistrationResult
from spatialwarp.qc import plot_overlay

Jupyter's default inline backend can't drive interactive `Slider`/click widgets — switch to a
real GUI backend so the interactive tools below actually open a window (`tk` ships with
Python; use `qt` instead if you have PyQt5/PySide and prefer it).

In [ ]:
%matplotlib tk

In [ ]:
# Set to True to actually click landmarks yourself in the interactive tools below (the real
# workflow). Set to False to run the notebook unattended using landmarks derived from this
# synthetic data's *known* ground truth — only possible here because we generated the data
# ourselves; with real data you don't have ground truth, so you'd always click.
INTERACTIVE = True

## 1. Build synthetic data

We create three things, mirroring a real MSI + Visium experiment:

- `moving_he_image` — the MSI section's own H&E scan (three Gaussian blobs + noise).
- `fixed_he_image` — a second, independent "Visium H&E" scan: the same blob pattern, but
  rotated, scaled, and shifted (simulating two different physical scans of related tissue).
- `msi_points_xy` / `msi_intensity` — MSI instrument raster points in an arbitrary raw
  coordinate system (large offset, unrelated to `moving_he_image`'s own pixel grid), whose
  intensities trace out the same blob pattern as `moving_he_image` once correctly placed.

We keep the ground-truth pixel mapping (`pixel_xy_true`) around purely so this tutorial can
show you the alignment error — you won't have this in real data.

In [ ]:
rng = np.random.default_rng(0)


def gaussian_blob(shape, center, sigma, amplitude=1.0):
    yy, xx = np.mgrid[0 : shape[0], 0 : shape[1]]
    return amplitude * np.exp(-(((xx - center[0]) ** 2 + (yy - center[1]) ** 2) / (2 * sigma**2)))


moving_shape = (400, 400)
moving_blob_centers = np.array([(100, 100), (300, 120), (200, 300)], dtype=float)
moving_he_image = sum(gaussian_blob(moving_shape, c, 25, 180) for c in moving_blob_centers)
moving_he_image = np.clip(moving_he_image + rng.normal(10, 5, moving_shape), 0, 255)

# fixed image: same pattern, rotated + scaled + shifted, different canvas size
theta, scale, shift = np.radians(15), 1.3, np.array([80, -30])
R = scale * np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
fixed_shape = (500, 500)
moving_center = np.array(moving_shape[::-1]) / 2
fixed_center = np.array(fixed_shape[::-1]) / 2
fixed_blob_centers = np.array([R @ (c - moving_center) + fixed_center + shift for c in moving_blob_centers])
fixed_he_image = sum(gaussian_blob(fixed_shape, c, 25 * scale, 180) for c in fixed_blob_centers)
fixed_he_image = np.clip(fixed_he_image + rng.normal(10, 5, fixed_shape), 0, 255)

# MSI raster: 40x40 grid, arbitrary raw instrument coordinates, mapped onto moving_he_image
# via a second (smaller) known rotation+scale -- this is the misalignment grid_align must undo.
gx, gy = np.meshgrid(np.arange(40), np.arange(40))
msi_pitch = 12.0
msi_origin = np.array([5000.0, 8000.0])
msi_points_xy = np.column_stack([gx.ravel() * msi_pitch + msi_origin[0], gy.ravel() * msi_pitch + msi_origin[1]])

theta2, scale2 = np.radians(8), moving_shape[0] / (40 * msi_pitch)
R2 = scale2 * np.array([[np.cos(theta2), -np.sin(theta2)], [np.sin(theta2), np.cos(theta2)]])
raw_center = msi_points_xy.mean(axis=0)
pixel_xy_true = (msi_points_xy - raw_center) @ R2.T + moving_center

px = np.clip(pixel_xy_true[:, 0].round().astype(int), 0, moving_shape[1] - 1)
py = np.clip(pixel_xy_true[:, 1].round().astype(int), 0, moving_shape[0] - 1)
msi_intensity = moving_he_image[py, px]

print("moving_he_image:", moving_he_image.shape, " fixed_he_image:", fixed_he_image.shape)
print("msi_points_xy:", msi_points_xy.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(moving_he_image, cmap="gray")
axes[0].set_title("moving_he_image (MSI section H&E)")
axes[1].imshow(fixed_he_image, cmap="gray")
axes[1].set_title("fixed_he_image (Visium H&E)")
plt.show()

## 2. Problem A: grid self-alignment

`run_grid_alignment` rasterizes `msi_points_xy`/`msi_intensity` onto their own native raster
grid (`rasterize_points`), then registers that pseudo-image against `moving_he_image`. Landmarks
seed the affine initialization — without them, registration has to converge from a cold start,
which is unreliable when the MSI raster is coarse/noisy relative to the real image.

Landmarks here are (point on `moving_he_image`, point on the *rasterized pseudo-image*), not on
the raw MSI coordinates — `pick_landmarks` shows you the pseudo-image for exactly this reason.

In [ ]:
pseudo_image, msi_transform = rasterize_points(msi_points_xy, msi_intensity)

if INTERACTIVE:
    # Click a point on the left (moving_he_image), then its match on the right (pseudo-image);
    # the 3 blob centers are an easy, well-spread set of landmarks to click. Close when done.
    grid_moving_landmarks, grid_fixed_landmarks = pick_landmarks(moving_he_image, pseudo_image)
else:
    # Fallback using known ground truth (see intro) -- not available with real data.
    msi_landmark_raw = np.array(
        [msi_points_xy[np.argmin(((pixel_xy_true - c) ** 2).sum(axis=1))] for c in moving_blob_centers]
    )
    grid_moving_landmarks = moving_blob_centers
    grid_fixed_landmarks = points_to_pixel(msi_landmark_raw, msi_transform)

aligned_xy = run_grid_alignment(
    image=moving_he_image,
    points_xy=msi_points_xy,
    values=msi_intensity,
    pick_landmarks_interactively=False,  # we already picked them above
    moving_landmarks=grid_moving_landmarks,
    fixed_landmarks=grid_fixed_landmarks,
    mesh_size=(4, 4),
    number_of_iterations=100,
)

error = np.linalg.norm(aligned_xy - pixel_xy_true, axis=1)
print(f"mean error vs ground truth: {error.mean():.2f}px (only measurable because this is synthetic)")

QC: overlay the aligned MSI points on `moving_he_image` and drag the alpha slider — they should
trace out the same three blobs as the underlying image.

In [ ]:
plot_overlay(moving_he_image, aligned_xy, values=msi_intensity)

## 3. Build SpatialData objects

`spatialwarp.pipeline.align()` operates on two `spatialdata.SpatialData` objects — each with one
image element and one table element whose `obsm['spatial']` holds pixel coordinates matching
that image. `examples/msi/build_msi_spatialdata.py` shows a real-data version of this for MSI
vendor exports; here we build both by hand to keep the tutorial self-contained.

In [ ]:
import anndata
from spatialdata import SpatialData
from spatialdata.models import Image2DModel, TableModel

# moving: the MSI dataset, now correctly placed on moving_he_image's pixel grid
n = len(aligned_xy)
moving_features = pd.DataFrame(
    {
        "metabolite_A": msi_intensity,
        "metabolite_B": msi_intensity[::-1] * 0.5 + rng.normal(0, 5, n),
    },
    index=[f"msi_{i}" for i in range(n)],
)
moving_adata = anndata.AnnData(X=moving_features.values.astype(np.float32), var=pd.DataFrame(index=moving_features.columns))
moving_adata.obsm["spatial"] = aligned_xy
moving_sdata = SpatialData(
    images={"he": Image2DModel.parse(np.moveaxis(np.atleast_3d(moving_he_image), -1, 0))},
    tables={"msi": TableModel.parse(moving_adata)},
)

# fixed: a Visium-like regular grid over fixed_he_image, with synthetic gene expression
gx2, gy2 = np.meshgrid(np.arange(20, 480, 15), np.arange(20, 480, 15))
fixed_xy = np.column_stack([gx2.ravel(), gy2.ravel()]).astype(float)
fpx = np.clip(fixed_xy[:, 0].round().astype(int), 0, fixed_he_image.shape[1] - 1)
fpy = np.clip(fixed_xy[:, 1].round().astype(int), 0, fixed_he_image.shape[0] - 1)
n2 = len(fixed_xy)
fixed_expression = pd.DataFrame(
    {
        "GeneA": fixed_he_image[fpy, fpx] + rng.normal(0, 5, n2),  # correlated with the blob pattern
        "GeneB": rng.normal(50, 10, n2),  # uncorrelated noise gene
    },
    index=[f"visium_{i}" for i in range(n2)],
)
fixed_adata = anndata.AnnData(X=fixed_expression.values.astype(np.float32), var=pd.DataFrame(index=fixed_expression.columns))
fixed_adata.obsm["spatial"] = fixed_xy
fixed_sdata = SpatialData(
    images={"he": Image2DModel.parse(np.moveaxis(np.atleast_3d(fixed_he_image), -1, 0))},
    tables={"visium": TableModel.parse(fixed_adata)},
)

moving_sdata, fixed_sdata

## 4. Problem B: cross-modality registration

Same recipe as grid self-alignment, just between two real images this time: pick landmarks, then
run landmark-guided elastic registration. Registration only depends on the two H&E images, not
the analyte, so if you had a second MSI modality (e.g. lipidomics) on this slide you'd compute
this once and reuse the `RegistrationResult` for both (see `examples/l12_walkthrough.ipynb`).

In [ ]:
if INTERACTIVE:
    cross_moving_landmarks, cross_fixed_landmarks = pick_landmarks(moving_he_image, fixed_he_image)
else:
    cross_moving_landmarks, cross_fixed_landmarks = moving_blob_centers, fixed_blob_centers

registration_result = register_elastic(
    moving_image=moving_he_image,
    fixed_image=fixed_he_image,
    moving_landmarks=cross_moving_landmarks,
    fixed_landmarks=cross_fixed_landmarks,
    mesh_size=(6, 6),
    number_of_iterations=100,
)
registration_result.save("tutorial_registration")  # RegistrationResult.load(...) to reuse later

QC: warp `moving_he_image` onto `fixed_he_image`'s grid and compare — the blobs should line up.

In [ ]:
warped_moving = registration_result.warp_image(moving_he_image)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(fixed_he_image, cmap="gray")
axes[0].set_title("fixed_he_image")
axes[1].imshow(warped_moving, cmap="gray")
axes[1].set_title("moving_he_image warped onto fixed's grid")
plt.show()

## 5. Full pipeline: `spatialwarp.align()`

This warps `fixed`'s points into `moving`'s pixel space using `registration_result`,
nearest-neighbor matches them against `moving`'s own points, and returns `fixed`'s table with
`moving`'s matched feature matrix merged into `obsm[moving_obsm_key]` — this is the same call
`examples/l12_walkthrough.ipynb` makes for real MSI/Visium data.

In [ ]:
merged = spatialwarp.align(
    moving=moving_sdata,
    fixed=fixed_sdata,
    registration_result=registration_result,
    distance_threshold=15.0,
    moving_obsm_key="metabolite",
)
print(f"kept {merged.n_obs} of {fixed_sdata.tables['visium'].n_obs} fixed points")
merged

In [ ]:
merged.obsm["metabolite"].head()

QC: the fixed points, warped into moving space (`obsm['spatial_warped']`), should land on top of
`moving_he_image` in the same pattern as the original MSI points.

In [ ]:
plot_overlay(moving_he_image, merged.obsm["spatial_warped"], values=merged.obsm["metabolite"]["metabolite_A"])

## 6. Quick correlation check

`GeneA` was built to correlate with the blob pattern (like `metabolite_A`); `GeneB` is pure
noise. A sanity-check correlation should show `metabolite_A` tracking `GeneA` but not `GeneB` —
downstream analysis like this is intentionally out of `spatialwarp`'s scope, but here's the
minimal version to confirm the alignment is actually meaningful, not just structurally correct.

In [ ]:
import scipy.stats

for gene in ["GeneA", "GeneB"]:
    gene_values = np.asarray(merged[:, gene].X).ravel()
    r, p = scipy.stats.pearsonr(gene_values, merged.obsm["metabolite"]["metabolite_A"])
    print(f"metabolite_A vs {gene}: r={r:.2f}, p={p:.2e}")

## Next steps

- `examples/l12_walkthrough.ipynb` — the same pipeline on a real MSI + Visium HD slide.
- `examples/msi/` — building a `SpatialData` object from real MSI vendor CSV exports.
- `spatialwarp.bunwarp` — if you have existing BUnwarpJ transform files from before this package
  switched to SimpleITK, this module still reads them (`read_bunwarp_matrix`, `warp_points`,
  `warp_image`) — it's just no longer used internally by `pipeline.align()`.